---
## Clean up

No Vector Search resources to clean up — the embeddings are stored in a Delta table (`main.default.ultrafeedback_embeddings`) which persists across assignments at no additional cost.


---
## 1. The human-in-the-loop *(Required)*

Automated judges are scalable but imperfect. They can miss domain nuances, apply criteria inconsistently, or disagree with expert opinions. **Human feedback** addresses this:

- A domain expert reviews agent outputs and rates them (good/bad, with rationale)
- The `align()` function uses this feedback to optimize the judge's prompt, making it better match human standards
- The aligned judge can then evaluate at scale, acting as a proxy for the human expert

This notebook aligns your custom judge to human feedback. The required path uses **SIMBA**, which revises the judge instructions based on examples where the current judge agrees or disagrees with your ratings.

### Requirements for alignment
- **Use the custom judge you created and registered in Assignment 5** (Section 8). In Assignment 5 you call `.register()` on your judge so it is stored in the experiment; here you load it with `get_scorer()`. Same name so your human feedback and `align()` target the same criteria.
- A judge created with `make_judge()` and registered (template-based)
- At least **10 traces** with human feedback (this notebook generates 12)
- The feedback assessment name must **exactly match** the judge name

---
## 2. Install dependencies *(Required)*

In [ ]:
%pip install --upgrade "mlflow>=3.9" databricks-langchain unitycatalog-ai[databricks] numpy databricks-agents backoff "dspy>=3.1.0"
dbutils.library.restartPython()

---
## 3. Recreate agent and custom judge *(Required)*

Load the agent from Assignment 4's `agent.py` (same approach as Assignment 5) and **load the custom judge you registered in Assignment 5** (Section 8) via `get_scorer()`. The judge you align in this assignment must be that same judge so your human feedback and `align()` apply to it. The embeddings table is verified first; then the agent is loaded via a wrapper so `agent_executor.invoke({"input": question})` works for trace generation.

### 3a. Verify embeddings table

The embeddings table (`main.default.ultrafeedback_embeddings`) was created in Assignment 3. Verify it exists and has the expected data.


In [ ]:
# Verify the embeddings table from Assignment 3 exists
emb_df = spark.table("main.default.ultrafeedback_embeddings")
print(f"Embeddings table has {emb_df.count()} rows.")
print(f"Columns: {emb_df.columns}")
display(emb_df.select("id", "instruction", "source").limit(3))


### 3b. Load agent from agent.py

Same approach as Assignment 5: load the Assignment 4 agent from `agent.py` (ensure it is in the same directory as this notebook or on the Python path). The wrapper exposes `agent_executor.invoke({"input": question})` returning `{"output": "..."}` for trace generation below.


In [ ]:
import mlflow

# Enable tracing for agent invocations (traces used in Section 4–5)
mlflow.langchain.autolog()

# Shared experiment across Assignments 4, 5, 6
EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/aai510_ultrafeedback_expert"
mlflow.set_experiment(EXPERIMENT_NAME)

# Load the agent from Assignment 4's agent.py (same as Assignment 5)
import agent
AGENT = agent.ToolCallingAgent(llm_endpoint=agent.LLM_ENDPOINT_NAME, tools=agent.TOOL_INFOS)


def _response_to_text(resp):
    """Extract final answer text from ResponsesAgentResponse."""
    text_parts = []
    for item in resp.output:
        c = getattr(item, "content", None)
        if c is None:
            continue
        for part in (c if isinstance(c, list) else [c]):
            if isinstance(part, dict) and "text" in part:
                text_parts.append(part["text"])
            elif hasattr(part, "text"):
                text_parts.append(part.text)
    return "\n".join(text_parts) if text_parts else str(resp.output)[:500]


# Wrapper so trace generation can use agent_executor.invoke({"input": question}) -> {"output": "..."}
class AgentExecutorWrapper:
    def invoke(self, inputs):
        resp = AGENT.predict({
            "input": [{"role": "user", "content": inputs["input"]}],
            "custom_inputs": {"session_id": "aai510-traces"},
        })
        return {"output": _response_to_text(resp)}


agent_executor = AgentExecutorWrapper()



In [ ]:
import mlflow
from mlflow.genai import get_scorer

# Load the custom judge you created and registered in Assignment 5 (Section 8)
# Use the same JUDGE_NAME you used when you registered the scorer in Assignment 5
JUDGE_NAME = "tool_usage_quality"  # Use your judge name from Assignment 5
EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/aai510_ultrafeedback_expert"

# Get the experiment ID (same experiment as Assignment 5) and load the scorer
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
custom_judge= get_scorer(name=JUDGE_NAME, experiment_id=experiment_id)

print(f"Loaded judge '{JUDGE_NAME}' from experiment.")

In [ ]:
# Quick test
resp = agent_executor.invoke({"input": "How many rows come from evol_instruct?"})
print("Agent test:", resp["output"][:200])

### 3c. Load the custom judge you registered in Assignment 5

**Load the custom judge you created and registered in Assignment 5** (Section 8). In Assignment 5 you called `.register()` on your judge so it is stored in the experiment. Here we load it with `get_scorer(name=JUDGE_NAME, experiment_id=...)`. Use the **same judge name** you used in Assignment 5 (e.g. `tool_usage_quality` or your own name). That judge is the one you will log feedback for (Section 5) and align (Section 6). The feedback assessment name when you log feedback must **exactly match** this judge's name.

**Docs:** [Versioning Scorers](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/versioning/)


## 4. Generate 12 traces *(Required)*

Run the agent on 12 diverse questions using agent_executor.invoke({"input": question}) (same interface as Assignment 5). Each invocation is logged as a trace in MLflow. Traces capture the full execution: inputs, outputs, tool calls, and intermediate steps.

The questions cover different tool types and edge cases to give you varied outputs to review.

In [ ]:
# 12 diverse questions for trace generation (target tools from Assignment 3)
TRACE_QUESTIONS = [
    # all_model_combinations
    "Which model combinations have the biggest disparity vs. chosen and rejected?",
    "Which model combinations have the most records? Show counts and average ratings.",
    # compare_models
    "How does gpt-4 compare to llama-2-7b-chat in the dataset?",
    "Head to head: gpt-4 vs wizardlm-7b. Which model was chosen more often?",
    # analyze_instruction
    "Analyze the complexity of: What is the weather today?",
    "Analyze this instruction: Design a comprehensive machine learning pipeline that ingests real-time streaming data, performs feature engineering, and deploys the best model to a REST API.",
    # Complex questions
    "Analyze this instruction: How many windows are there in the Empire State Building? Are there similar questions on this topic?",
    "Find instructions similar to wild rice, and which model is the best?",
    # search_similar_instructions
    "Find instructions in the dataset similar to 'python programming'. Show a few examples.",
    "Search for instructions about quantum computing.",
    "What instructions in the dataset mention health benefits?",
    # Multi-step or edge cases
    "Find instructions about writing code, then analyze the complexity of one of them.",
]

# Run all queries and store results with the official MLflow tagging
trace_results = []
import mlflow
for i, question in enumerate(TRACE_QUESTIONS):
    print(f"\n--- Trace {i+1}/12: {question[:60]}... ---")
    try:        
        response = agent_executor.invoke({"input": question})
        output = response["output"]
        print(f"Output: {output[:150]}...")
        trace_results.append({
            "question": question,
            "output": output,
            "error": None
        })
    except Exception as e:
        print(f"Error: {e}")
        trace_results.append({
            "question": question,
            "output": None,
            "error": str(e)
        })

print(f"\n=== Generated {len(trace_results)} traces ===")

In [ ]:
# Retrieve trace IDs from MLflow - get the most recent 12
# This is just here so you don't have to keep rerunning the trace creation if you come back to this after the notebook resets
traces = mlflow.search_traces(
    max_results=12,
    order_by=["timestamp_ms DESC"],
)

print(f"Found {len(traces)} recent traces in experiment.")


In [ ]:
# Assuming 'traces' already contains all relevant rows/columns
# If your 'inputs' column isn't named 'inputs' or 'response', rename as needed
answer_sheet_df = traces.rename(columns={'response': 'output'})  # MLflow usually expects 'output'
# If 'inputs' is nested, flatten as needed

result = mlflow.genai.evaluate(
    data=answer_sheet_df,
    scorers=[custom_judge] 
)

In [ ]:
# Tag each trace for alignment - REQUIRED for later filtering
import mlflow

trace_ids = traces['trace_id'].values

for trace_id in trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="eval", value="align")

print(f"Tagged {len(traces)} traces with eval: align tag.")

---
## 5. Review traces and log feedback *(Required)*

Now you become the **domain expert**. For each trace:

1. **Review the trace** in the MLflow UI (Experiments > your experiment > Traces tab). Look at:
   - Did the agent call the right tool?
   - Was the response accurate?
   - Did the agent handle edge cases well?

2. **Log feedback** using the MLflow experiments UI.
   - Choose the **same judge name** and feedback type as your custom judge.
   - Provide a rating that reflects your real assessment of the trace.
   - Add a short rationale for each label so the alignment step has useful signal.
   - A good alignment set usually includes both positive and negative examples. If all 12 traces receive the same label, revisit your trace set or your labeling criteria.

**Important:** The assessment name must exactly match the judge name for `align()` to work. Use the same judge you recreated in Section 3c.

**Docs:** [Label during development](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/dev-annotations) · [Feedback collection](https://mlflow.org/docs/latest/genai/assessments/feedback/)

---
## 6. Align the judge *(Required)*

The `align()` function improves **the custom judge you created in Assignment 5** (loaded in Section 3c) based on your human feedback. This notebook uses **SIMBAAlignmentOptimizer**, which iteratively proposes instruction updates and keeps the ones that best match your labels. This is the only required alignment path in this assignment.

**Requirements:**
- The judge you recreated in Section 3c (same as your Assignment 5 custom judge)
- At least 10 traces with feedback (we have 12), tagged with `eval='align'`
- Feedback assessment name matches the judge name
- MLflow 3.9+ for alignment optimizers

**Docs:** [Align judges with humans](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/align-judges) · [SIMBA](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/simba/) · [MemAlign](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/memalign/)

If you are curious about the alternative memory-based approach, read the MemAlign docs above. It is not part of the required assignment flow.


In [ ]:
# Retrieve the labeled traces that will be used for alignment.
traces_for_alignment = mlflow.search_traces(
    filter_string="tag.eval = 'align'",
    max_results=12,
    return_type="list",
)

# Keep the original trace objects because alignment expects full trace metadata.
print(f"Found {len(traces_for_alignment)} traces with tag eval='align' in experiment.")

# Print out SME & LLM Judge Assessments

Run the cell below before alignment. It prints the judge assessment and the matching human assessment for each trace so grading can quickly verify that:

- the trace already has an LLM judge assessment
- the student logged human feedback against the same judge name
- both assessments are present on the same alignment set

If any trace shows `(missing)`, the feedback step was not completed correctly and the student should fix Section 5 before continuing.

In [ ]:
# Print the LLM judge assessment and the matching human feedback for quick grading.
def _to_dict_if_possible(obj):
    if isinstance(obj, dict):
        return obj
    for method_name in ("to_dict", "model_dump", "dict"):
        if hasattr(obj, method_name):
            try:
                return getattr(obj, method_name)()
            except Exception:
                pass
    return None


def _get_value(obj, *names):
    if obj is None:
        return None
    if isinstance(obj, dict):
        for name in names:
            if name in obj:
                return obj[name]
        return None
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    obj_dict = _to_dict_if_possible(obj)
    if isinstance(obj_dict, dict):
        for name in names:
            if name in obj_dict:
                return obj_dict[name]
    return None


def _normalize_text(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return str(value)
    value_dict = _to_dict_if_possible(value)
    if isinstance(value_dict, dict):
        for key in ("value", "boolean_value", "string_value", "text"):
            if key in value_dict:
                return str(value_dict[key])
    return str(value)


def _extract_assessments(trace):
    trace_dict = _to_dict_if_possible(trace) or {}
    candidates = [
        _get_value(trace, "assessments"),
        _get_value(_get_value(trace, "info"), "assessments"),
        _get_value(_get_value(trace, "data"), "assessments"),
        trace_dict.get("assessments"),
    ]
    if isinstance(trace_dict.get("info"), dict):
        candidates.append(trace_dict["info"].get("assessments"))
    if isinstance(trace_dict.get("data"), dict):
        candidates.append(trace_dict["data"].get("assessments"))

    for candidate in candidates:
        if candidate:
            return list(candidate)
    return []


def _extract_question(trace):
    trace_dict = _to_dict_if_possible(trace) or {}
    candidates = [
        _get_value(trace, "inputs"),
        _get_value(_get_value(trace, "data"), "inputs"),
        _get_value(_get_value(_get_value(trace, "data"), "request"), "inputs"),
        trace_dict.get("inputs"),
    ]
    if isinstance(trace_dict.get("data"), dict):
        candidates.append(trace_dict["data"].get("inputs"))
        if isinstance(trace_dict["data"].get("request"), dict):
            candidates.append(trace_dict["data"]["request"].get("inputs"))

    for candidate in candidates:
        if isinstance(candidate, dict):
            if "input" in candidate:
                return str(candidate["input"])
            if candidate:
                first_value = next(iter(candidate.values()))
                return str(first_value)
        if isinstance(candidate, str):
            return candidate
    return None


def _extract_trace_id(trace):
    return (
        _get_value(trace, "trace_id")
        or _get_value(_get_value(trace, "info"), "trace_id")
        or (_to_dict_if_possible(trace) or {}).get("trace_id")
        or "unknown-trace"
    )


def _assessment_bucket(assessment):
    source = _get_value(assessment, "source_type", "source", "source_id")
    source_name = _normalize_text(source).lower() if source is not None else ""
    if any(token in source_name for token in ("human", "manual", "feedback", "annotation", "label", "user")):
        return "human"
    return "llm"


def _assessment_summary(assessment):
    value = _normalize_text(_get_value(assessment, "value", "feedback", "result")) or "(missing)"
    rationale = _normalize_text(_get_value(assessment, "rationale", "justification", "explanation")) or "(missing)"
    return value, rationale


print("\n=== Pre-alignment assessment check ===")
llm_present_count = 0
human_present_count = 0
both_present_count = 0
for idx, trace in enumerate(traces_for_alignment, start=1):
    relevant_assessments = [
        assessment
        for assessment in _extract_assessments(trace)
        if (_get_value(assessment, "name", "assessment_name", "key") or JUDGE_NAME) == JUDGE_NAME
    ]

    llm_assessment = next((a for a in relevant_assessments if _assessment_bucket(a) == "llm"), None)
    human_assessment = next((a for a in relevant_assessments if _assessment_bucket(a) == "human"), None)

    question = _extract_question(trace)
    question_preview = (question[:100] + "...") if question and len(question) > 100 else question

    print(f"\nTrace {idx}: {_extract_trace_id(trace)}")
    if question_preview:
        print(f"  Question: {question_preview}")

    llm_value, llm_rationale = _assessment_summary(llm_assessment)
    human_value, human_rationale = _assessment_summary(human_assessment)

    if llm_assessment is not None:
        llm_present_count += 1
    if human_assessment is not None:
        human_present_count += 1
    if llm_assessment is not None and human_assessment is not None:
        both_present_count += 1

    print(f"  LLM assessment: {llm_value}")
    print(f"  LLM rationale: {llm_rationale}")
    print(f"  Human assessment: {human_value}")
    print(f"  Human rationale: {human_rationale}")

print("\n=== Assessment completeness summary ===")
print(f"Traces with LLM assessments: {llm_present_count}/{len(traces_for_alignment)}")
print(f"Traces with human assessments: {human_present_count}/{len(traces_for_alignment)}")
print(f"Traces with both present: {both_present_count}/{len(traces_for_alignment)}")


In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
    stream=sys.stdout,
)

# Keep only SIMBA-related logs. Switch any of these to DEBUG if you need deeper troubleshooting.
logging.getLogger("SIMBAAlignmentOptimizer").setLevel(logging.INFO)
logging.getLogger("mlflow.genai.judges.optimizers.simba").setLevel(logging.INFO)
logging.getLogger("dspy.teleprompt.simba").setLevel(logging.INFO)

# Silence noisy internals
for name in [
    "mlflow",
    "mlflow.genai.judges.optimizers.dspy_utils",
    "databricks.sdk",
    "httpcore",
    "httpx",
    "urllib3",
]:
    logging.getLogger(name).setLevel(logging.WARNING)


In [ ]:
# Required path: align the judge with SIMBA.
# Expect this to take several minutes. If you hit rate limits, rerun the cell or reduce max_steps.
# Alignment may improve the judge, but it is not guaranteed to outperform your original prompt on every trace.

# Align the judge using traces with SME feedback
from mlflow.genai.judges import make_judge
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer
optimizer = SIMBAAlignmentOptimizer(
    model="databricks:/databricks-gpt-oss-120b",  # Optimizer model used to propose judge revisions.
    simba_kwargs={
        "max_steps": 2,  # Each step runs another optimization round. Lower is faster.
        "max_demos": 0,  # Rule-only optimization. Set >0 to let SIMBA add few-shot demos.
    },
)
aligned_judge = custom_judge.align(traces=traces_for_alignment, optimizer=optimizer)
print(f"Alignment complete for judge: {aligned_judge.name}")

In [ ]:
print('ORIGINAL JUDGE:\n')
print(custom_judge.instructions)
print('\n')
print("_"*100)
print("\nALIGNED JUDGE\n")
print(aligned_judge.instructions)

### Compare the original and aligned judges

Rerun evaluation on the same traces so you can compare how the **original** and **aligned** judges score identical outputs.

In [ ]:
print("\nRunning aligned judge for comparison...")
traces_for_eval = mlflow.search_traces(
    filter_string="tag.eval = 'align'",
    max_results=12,
)
answer_sheet_df_align = traces_for_eval.rename(columns={'response': 'output'})  # MLflow usually expects 'output'

aligned_results = mlflow.genai.evaluate(
    data=answer_sheet_df_align,
    scorers=[aligned_judge]
)


---
## 7. Reflection *(Required)*

### 7a. How did alignment change the judge? Did it reflect your manually provided feedback?

*Compare the original and aligned judge prompts. What changed? Did the aligned judge add new criteria, change the emphasis, or use different language? Did the new evaluation job lead to different results?*

**[Your answer here]**

### 7b. What does this teach about human-in-the-loop evaluation?

*Reflect on the entire arc: subjective reviews (Week 4) → automated judges (Week 5) → human feedback → alignment (this week). How does this connect to the principle of humans steering agents?*

**[Your answer here]**

*The required assignment ends after you answer these two reflection prompts. Everything below is optional enrichment.*

---
## Optional Extension: optimize_anything and agent skills

*This section is not required for submission and is not part of the grading rubric.*

**What it is:** [GEPA](https://gepa-ai.github.io/gepa/)'s **optimize_anything** API optimizes *any* text artifact — prompts, code, configs, or **agent skills** — using an evaluator and LLM-driven reflection. You declare what to optimize and how to measure it; the system handles the search (single-task, multi-task, or generalization). Unlike hand-written mutation prompts, you provide a **seed** (or a natural-language objective), an **evaluator** that scores candidates and can return **Actionable Side Information (ASI)** (e.g. error messages, execution traces), and optionally a **dataset** and **valset** so the optimized artifact generalizes to unseen examples.

**Why agent skills matter:** Agent **skills** are natural-language instructions and best practices that help an agent perform well on a specific codebase or domain. They are text artifacts: if you can run the agent on real tasks and score outcomes, you can optimize them. In practice, GEPA-optimized skills have been shown to push coding agents to near-perfect task completion while being **47% faster** and to **transfer across models** (e.g. skills learned for one repo improving another). So learning to generate or refine agent skills with optimize_anything is a natural next step after judge alignment: you go from "align a judge to human feedback" to "optimize the instructions the agent follows so it succeeds on more tasks."

**Giving it an objective:** You don't have to provide a starting artifact. You can call `optimize_anything` with an **objective** — a natural-language description of what you want — and no `seed_candidate`. The reflection LLM then bootstraps the first candidate from that description ("seedless" mode). For example: *"Generate agent skills (natural-language instructions) that help the UltraFeedback Expert use its tools appropriately on real user questions and generalize to unseen tasks."* The evaluator still runs your agent on a dataset and returns scores (and ASI); the optimizer proposes and refines candidates until they meet your objective. This is especially useful when the solution space is large or you're not sure what a good seed looks like.

**How to get started:**

1. **Install:** `pip install gepa` (must be >=0.1.0)
2. **Define an evaluator** that takes a candidate (e.g. a skill document or prompt string), runs your agent on a set of tasks, and returns a score. The evaluator **produces** ASI: either return a `(score, side_info)` tuple or call `oa.log()` inside the evaluator so the optimizer has diagnostics (errors, traces) to guide reflection.
3. **Call** `optimize_anything(evaluator=evaluate, objective="...", dataset=..., valset=...)` with an **objective** and no seed to go seedless, or pass `seed_candidate=...` if you have a starting draft. Use **generalization** mode (dataset + valset) so the optimized skill transfers to unseen tasks.

For full examples (including the agent-skills case study), code walkthroughs, and the three optimization modes, use the links below.

**Where to go next:**

- **Blog — optimize_anything intro (agent skills, cloud algorithms, ARC-AGI, etc.):** [Introducing optimize_anything](https://gepa-ai.github.io/gepa/blog/2026/02/18/introducing-optimize-anything/)
- **GEPA site (quick start, adapters, tutorials):** [gepa-ai.github.io/gepa](https://gepa-ai.github.io/gepa/)
- **Repo and API reference:** [github.com/gepa-ai/gepa](https://github.com/gepa-ai/gepa)



---
## Lab complete

### Required (Sections 1-7)
- [ ] **Section 3:** Agent and custom judge recreated. Vector Search working.
- [ ] **Section 4:** 12 traces generated and visible in MLflow Experiments.
- [ ] **Section 5:** Human feedback logged for all 12 traces with honest ratings and rationales.
- [ ] **Section 6:** SIMBA alignment ran successfully. Original and aligned judge were compared on the same traces.
- [ ] **Section 7:** Both reflection questions answered.
- [ ] **Cleanup note:** Reviewed the cleanup note at the top of the notebook. No additional cleanup is required for this lab.

**Submit:** Your executed notebook (`.ipynb` with all outputs) and the completed `SUBMISSION_6.md`.

*You have completed the required assignment flow. The optimize_anything section above is optional enrichment and is not required for submission.*